## Berechnen der Metrik-Werte und Signifikanzprüfung

In diesem Skript werden die vom Adapter_Generator Notebook erzeugten Prediction-JSONLs verwendet, um die Metrik-Werte MAP@100 und recall@k zu berechnen und die erreichten Deltas mittels des einseitigen t-Tests auf Signifikanz zu überprüfen.

In [ ]:
import json
import numpy as np
from scipy.stats import ttest_rel

# ------------------------ Hilfsfunktionen -------------------------------

from functions.load_topk_docids_from_predictions_jsonl import load_topk_docids_from_predictions_jsonl
from functions.load_single_gold_map import load_single_gold_map
from functions.ranks_of_gold import ranks_of_gold
from functions.single_positive_metrics_from_ranks import single_positive_metrics_from_ranks
from functions.paired_ttest_one_sided_greater import paired_ttest_one_sided_greater

# ------------------------ Anwendung -------------------------------

KS = (1, 5, 10, 20, 50)
K_MAX = 100

prediction_file_name = "Information-Retrieval_evaluation_myEvaluator_predictions_cosine.jsonl"
baseline_jsonl = "results/predictions/baseline/intfloat__multilingual-e5-base/" + prediction_file_name
adapted_modell_jsonl      = "results/manual/baseline/pfeiffer_r16_intfloat__multilingual-e5-base__S314/" + prediction_file_name

# 1) Dictionary mit den sortierten chunk_ids pro Frage
run_base = load_topk_docids_from_predictions_jsonl(baseline_jsonl)
run_adapted  = load_topk_docids_from_predictions_jsonl(adapted_modell_jsonl)

# 2) gold-Map erstellen
gold_map = load_single_gold_map("datasets/GerManualDPR/manual/test_dataset.json")

# Sicherstellen, dass stets alle querys enthalten sind
print(f"\nCoverage: gold={len(gold_map)}, base_run={len(run_base)}, adapted_run={len(run_adapted)}")

# 3) Identifizieren der Rangposition der Gold-Passage zu der jeweiligen Frage
ranks_base = ranks_of_gold(run_base, gold_map, k_max=K_MAX)
ranks_adapted  = ranks_of_gold(run_adapted,  gold_map, k_max=K_MAX)

# 4) Berechnen der Metrik-Werte
recall_base, ap100_base = single_positive_metrics_from_ranks(ranks_base, ks=KS)
recall_adapted,  ap100_adapted  = single_positive_metrics_from_ranks(ranks_adapted,  ks=KS)

# 5) Ausgabe der aggregierten Metriken
print("\nAggregated metrics (mean over queries)")
for k in KS:
    print(f"Recall@{k:2d} (base/adapted): {np.mean(list(recall_base[k].values())):.4f}  /  {np.mean(list(recall_adapted[k].values())):.4f}")

print(f"MAP@100   (base/adapted): {np.mean(list(ap100_base.values())):.4f}  /  {np.mean(list(ap100_adapted.values())):.4f}")

# 6) Durchführung und Ausgabe des einseitigen, gepaarten t-Tests
print("\nPaired one-sided t-tests (H1: adapted > base)")
for k in KS:
    res = paired_ttest_one_sided_greater(recall_base[k], recall_adapted[k])
    print(f"Recall@{k:2d}: p_one={res['p']:.4g}, t={res['t']:.3f}, mean_diff={res['mean_diff']:.6f}, n={res['n']}")

res_map = paired_ttest_one_sided_greater(ap100_base, ap100_adapted)
print(f"MAP@100  : p_one={res_map['p']:.4g}, t={res_map['t']:.3f}, mean_diff={res_map['mean_diff']:.6f}, n={res_map['n']}")
